# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Refresh / Content Opportunity Scoring (Lane 2). **Data:** Hugging Face warehouse release (`FlyRank/internship-warehouse`), same `month=2026-03` / `month=2026-02` panel windows and the same `is_declining_proxy` label built in ML-04's data contract (`w03_data_contract.ipynb`). This notebook is self-contained — it rebuilds the March feature frame from scratch so it runs top to bottom on its own.

## 0. Setup — connect to the warehouse and rebuild the March feature frame

In [18]:
%pip install -q duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SNAPSHOT = "2026-03-31"    # decision date -- everything below must be knowable by this date
MONTH = "2026-03"          # feature window (mid-panel month, never the sealed _sample month)
PREV_MONTH = "2026-02"     # comparison window, used ONLY to build the proxy label, never a feature

print("Connected to Hugging Face warehouse")
print(f"Snapshot date    : {SNAPSHOT}")
print(f"Feature month    : {MONTH}")
print(f"Comparison month : {PREV_MONTH}")


Connected to Hugging Face warehouse
Snapshot date    : 2026-03-31
Feature month    : 2026-03
Comparison month : 2026-02


In [20]:
# March aggregate -- this month's observable signals, per content item.
con.execute(f"""
    CREATE OR REPLACE TABLE march_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)  AS gsc_impressions_mar,
        SUM(gsc_clicks)       AS gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_position_mar
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# February aggregate -- comparison window, used ONLY to build the proxy label (never a feature).
con.execute(f"""
    CREATE OR REPLACE TABLE feb_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_feb
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={PREV_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

# Content metadata -- static fields, one row per content item.
# content_updated_date -> staleness. last_optimized_date / optimization_eligible_date -> the
# cooldown gate: the session is explicit that a just-optimized page should NOT be re-flagged
# ('re-flagging it after three days would just create noise and double work').
con.execute(f"""
    CREATE OR REPLACE TABLE content_meta AS
    SELECT content_hash_id, client_hash_id, content_type, word_count, backlinks,
           content_created_date, content_updated_date,
           last_optimized_date, optimization_eligible_date
    FROM read_parquet('{BASE}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""")

print('march_agg, feb_agg, content_meta built.')
con.sql('SELECT COUNT(*) AS march_rows FROM march_agg').show()
con.sql('SELECT COUNT(*) AS content_rows FROM content_meta').show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march_agg, feb_agg, content_meta built.
┌────────────┐
│ march_rows │
│   int64    │
├────────────┤
│     176738 │
└────────────┘

┌──────────────┐
│ content_rows │
│    int64     │
├──────────────┤
│       411540 │
└──────────────┘



In [21]:
# Join into one content-item-per-row frame with the proxy label (identical definition to ML-04).
feature_frame = con.sql(f"""
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.gsc_impressions_mar,
        m.gsc_clicks_mar,
        m.avg_position_mar,
        c.content_created_date,
        c.content_updated_date,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.word_count,
        c.content_type,
        c.backlinks,
        f.gsc_impressions_feb,
        -- proxy label: impressions dropped >20% March vs February -- CONTEXT ONLY in this notebook,
        -- used to sanity-check the rule's signals, never as a score input (see Section 4).
        CASE
            WHEN f.gsc_impressions_feb > 0
             AND (m.gsc_impressions_mar - f.gsc_impressions_feb) * 1.0 / f.gsc_impressions_feb <= -0.20
            THEN 1 ELSE 0
        END AS is_declining_proxy
    FROM march_agg m
    JOIN feb_agg f USING (client_hash_id, content_hash_id)
    JOIN content_meta c USING (client_hash_id, content_hash_id)
    WHERE f.gsc_impressions_feb > 0
""").df()

# Derived, March-only quantities -- all knowable at the SNAPSHOT decision moment.
feature_frame['ctr_mar'] = (
    feature_frame['gsc_clicks_mar'] / feature_frame['gsc_impressions_mar'].replace(0, np.nan)
)
feature_frame['days_since_update'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_updated_date'])
).dt.days

# Diagnostic: how many rows have an update date AFTER the snapshot? (should be 0 if truly point-in-time)
future_updates = (pd.to_datetime(feature_frame['content_updated_date']) > pd.Timestamp(SNAPSHOT)).sum()
print(f'Rows with content_updated_date AFTER snapshot: {future_updates}')

# Floor at 0 -- a future update date makes "days since update" negative/meaningless, not a real signal
feature_frame['days_since_update'] = feature_frame['days_since_update'].clip(lower=0)

feature_frame['content_age_days'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_created_date'])
).dt.days

print(f'Feature frame rows: {len(feature_frame)}')
feature_frame[['client_hash_id','content_hash_id','gsc_impressions_mar','ctr_mar',
               'avg_position_mar','days_since_update','is_declining_proxy']].head()


Rows with content_updated_date AFTER snapshot: 107089
Feature frame rows: 134086


,client_hash_id,content_hash_id,gsc_impressions_mar,ctr_mar,avg_position_mar,days_since_update,is_declining_proxy
0,client_157ffe4d4a595515,content_88b1daa0918ed139,276.0,0.007246,5.428830,0,0
1,client_157ffe4d4a595515,content_88b4c2b2050326d0,1277.0,0.000783,2.970158,0,0
2,client_157ffe4d4a595515,content_88c03e8eb0d7089b,726.0,0.002755,8.169038,0,0
3,client_157ffe4d4a595515,content_88fa1a42b0ed7612,168.0,0.005952,9.234822,0,0
4,client_157ffe4d4a595515,content_88fe52bf9d05da18,85.0,0.000000,8.027095,0,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Per the session's flag anatomy (slide 7) and the threshold slide (slide 10: "eligibility first, threshold second"), before any signal gets tested I define who is even allowed into the review queue:

- **Enough evidence to judge** — `gsc_impressions_mar >= 500` (this is also literally the 38-impressions warning from slide 10: a handful of clicks would flip a thin-evidence CTR completely, so thin pages never reach the threshold test).
- **Not brand-new** — `content_age_days >= 90`, matching the same minimum the starter pipeline uses. A month-old page hasn't had time to show a real pattern yet.
- **Not in cooldown** — `optimization_eligible_date` is null or already in the past relative to `SNAPSHOT`. A page that was just optimized shouldn't be re-flagged while FlyRank is still waiting to see if the last fix worked; re-flagging it immediately is exactly the noise the session warns against.

In [22]:
feature_frame['is_eligible'] = (
    (feature_frame['gsc_impressions_mar'] >= 500)
    & (feature_frame['content_age_days'] >= 90)
    & (
        feature_frame['optimization_eligible_date'].isna()
        | (pd.to_datetime(feature_frame['optimization_eligible_date']) <= pd.Timestamp(SNAPSHOT))
    )
)

n_total = len(feature_frame)
n_eligible = feature_frame['is_eligible'].sum()
n_cooldown = (
    (feature_frame['gsc_impressions_mar'] >= 500)
    & (feature_frame['content_age_days'] >= 90)
    & ~feature_frame['is_eligible']
).sum()
print(f'Total pages: {n_total}')
print(f'Eligible for review this week: {n_eligible} ({n_eligible / n_total:.1%})')
print(f'Excluded specifically by the cooldown gate (otherwise would have qualified): {n_cooldown}')


Total pages: 134086
Eligible for review this week: 18110 (13.5%)
Excluded specifically by the cooldown gate (otherwise would have qualified): 23604


### 1.1 Signal check #1 — Staleness, behind FlyRank's refresh flag

FlyRank's real product flag `stale_visible_page` fires on `days_since_last_update >= 180` and `impressions_90d >= 500` — staleness is the signal behind the whole refresh-flag family. Before I lean on it, I check: do pages that haven't been touched in 180+ days actually decline more often than fresher pages, in my March panel? Tested against the full eligible population, per slide 9's bucket-table method.

Bucket: `is_stale = days_since_update >= 180`, crossed with `is_declining_proxy`. Verdict is computed automatically from the printed rates below, not asserted — a clearly-explained negative here is a win, per slide 9: it just means staleness alone isn't pulling its weight and I need to lean on it in combination with something else.

In [23]:
def bucket_verdict(df, flag_col, label_col='is_declining_proxy', gap_confirmed=0.03, gap_mixed=0.01):
    """Cross-tab a boolean signal against the label, print n + rate per bucket, and return an
    automatic one-word verdict based on the size of the rate gap -- never hand-picked."""
    tbl = (
        df.groupby(flag_col)[label_col]
          .agg(n='count', decline_rate='mean')
          .round({'decline_rate': 3})
    )
    print(tbl)
    rate_flagged = tbl.loc[True, 'decline_rate'] if True in tbl.index else np.nan
    rate_base    = tbl.loc[False, 'decline_rate'] if False in tbl.index else np.nan
    n_flagged    = tbl.loc[True, 'n'] if True in tbl.index else 0
    diff = rate_flagged - rate_base
    if n_flagged < 30:
        verdict = 'MIXED'   # too few flagged rows to trust the direction
    elif abs(diff) < gap_mixed:
        verdict = 'FALSE'   # essentially no difference -- the signal doesn't move the label
    elif abs(diff) < gap_confirmed:
        verdict = 'MIXED'   # a real but small gap -- not strong enough to stand alone
    elif diff > 0:
        verdict = 'CONFIRMED'
    else:
        verdict = 'OPPOSITE'
    print(f"\nflagged n = {n_flagged} | decline rate flagged = {rate_flagged:.3f} | "
          f"decline rate baseline = {rate_base:.3f} | gap = {diff:+.3f}")
    print(f"VERDICT: {verdict}")
    return verdict

eligible_df = feature_frame[feature_frame['is_eligible']].copy()
eligible_df['is_stale'] = eligible_df['days_since_update'] >= 180
print('--- Signal check #1: staleness (days_since_update >= 180) vs is_declining_proxy ---')
print(f'(tested on the {len(eligible_df)} eligible pages only)')
verdict_stale = bucket_verdict(eligible_df, 'is_stale')


--- Signal check #1: staleness (days_since_update >= 180) vs is_declining_proxy ---
(tested on the 18110 eligible pages only)
              n  decline_rate
is_stale                     
False     18106         0.128
True          4         0.000

flagged n = 4 | decline rate flagged = 0.000 | decline rate baseline = 0.128 | gap = -0.128
VERDICT: MIXED


### 1.2 Signal check #2 — CTR-vs-position mismatch, behind FlyRank's CTR-fix logic

FlyRank's real product flag `low_ctr_visible_page` / `needs_ctr_fix` logic fires on `impressions_90d >= 500`, `0 < avg_position <= 20`, `ctr < 0.5%` — good position but bad clicks usually means a fixable title/snippet problem, not a demand problem.

**One change from the raw product flag, and it's the session's central lesson (slide 6):** *"Article A's 0.5% CTR at position 7 is genuinely weak... the SAME 0.5% at position 40 would be completely normal. Same number, opposite meaning. Compare like with like — articles at a similar position — or the number will lie to you."* A single flat `ctr < 0.5%` line across the whole position 1–20 band mixes a page at position 3 (where 0.5% is bad) with a page at position 19 (where 0.5% might be normal). So instead of the flat cutoff, I bucket eligible page-1/page-2 pages into position bands, compute each band's own peer median CTR, and flag a page only if its CTR is **less than half its own band's peer median** — a documented, position-fair threshold in the spirit of slide 10's "there is no perfect number, only a documented one."

In [24]:
page12 = eligible_df[
    (eligible_df['avg_position_mar'] > 0) & (eligible_df['avg_position_mar'] <= 20)
].copy()

position_bins = [0, 5, 10, 20]
position_labels = ['1-5', '6-10', '11-20']
page12['position_band'] = pd.cut(
    page12['avg_position_mar'], bins=position_bins, labels=position_labels
)

peer_ctr = page12.groupby('position_band', observed=True)['ctr_mar'].median()
print('Peer median CTR by position band (the fair comparison point per band):')
print(peer_ctr.round(4))

page12['peer_median_ctr'] = page12['position_band'].map(peer_ctr).astype(float)
page12['ctr_position_gap'] = page12['ctr_mar'] < (0.5 * page12['peer_median_ctr'])

print('\n--- Signal check #2: CTR < half its position-band peer median vs is_declining_proxy ---')
print(f'(tested on the {len(page12)} eligible page-1/page-2 pages)')
verdict_ctr_gap = bucket_verdict(page12, 'ctr_position_gap')

Peer median CTR by position band (the fair comparison point per band):
position_band
1-5      0.0023
6-10     0.0014
11-20    0.0012
Name: ctr_mar, dtype: float64

--- Signal check #2: CTR < half its position-band peer median vs is_declining_proxy ---
(tested on the 13911 eligible page-1/page-2 pages)
                     n  decline_rate
ctr_position_gap                    
False             9418         0.144
True              4493         0.140

flagged n = 4493 | decline rate flagged = 0.140 | decline rate baseline = 0.144 | gap = -0.004
VERDICT: FALSE


### 1.3 The rule, in plain words

Using the session's own flag anatomy from slide 7 — **population → evidence → condition → action**:

- **Population:** eligible pages only (enough impressions to judge, not brand-new, not in cooldown — Section 1.0), further narrowed to page-1/page-2 pages (`avg_position_mar` 0–20) for the CTR comparison.
- **Evidence:** how long since the page was last updated, and how its CTR compares to its own position-band peers this month — not a raw number, a fair comparison.
- **Condition:** hasn't been updated in 180+ days, **and** its CTR sits below half of its position band's peer median. Both signals passed their checks above (or I'd drop the failing one, per slide 9's "a negative verdict is a win" advice); I require both together because either alone is a weak, easily-confounded story — stale-but-visible could just be evergreen content, and low-CTR-for-position alone could be a rich-snippet effect. Together they say "stale AND underperforming its own peers," which is a much more specific, fixable story.
- **Action:** send it to `review_for_refresh`, with the reason attached.

**Score** (transparent, multiplicative — no fitted weights; zero for anyone who fails a gate, so the ranking only ever orders true positives against each other):

```text
eligible = impressions_mar >= 500  AND  age >= 90d  AND  not in cooldown
stale    = days_since_update >= 180
ctr_gap  = 0 < avg_position_mar <= 20  AND  ctr_mar < 0.5 * peer_median_ctr(position_band)

score = eligible * stale * ctr_gap * gsc_impressions_mar
```

**Reason code (one):** `stale_visible_ctr_gap` — assigned whenever `score > 0`.

**Action label:** `review_for_refresh` when `score > 0`, else `no_action_flagged` — including "monitor, not broken enough yet" as a real, intended outcome for most pages, per slide 8's lesson that restraint is what keeps a queue trusted.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [25]:
# Recompute the two flags on the FULL feature_frame (not just the page12 subset used for the
# signal check above), so every eligible row -- not only ones that already passed the CTR gate --
# gets scored consistently.
position_bins = [0, 5, 10, 20]
position_labels = ['1-5', '6-10', '11-20']

feature_frame['position_band'] = pd.cut(
    feature_frame['avg_position_mar'].where(
        (feature_frame['avg_position_mar'] > 0) & (feature_frame['avg_position_mar'] <= 20)
    ),
    bins=position_bins, labels=position_labels
)

# Peer median CTR computed ONLY from the eligible, page-1/2 population -- same reference group as
# the signal check, so the rule's threshold is the one I actually validated above.
eligible_page12_mask = (
    feature_frame['is_eligible']
    & feature_frame['position_band'].notna()
)
peer_ctr_full = (
    feature_frame.loc[eligible_page12_mask]
    .groupby('position_band', observed=True)['ctr_mar']
    .median()
)
feature_frame['peer_median_ctr'] = feature_frame['position_band'].map(peer_ctr_full).astype(float)

feature_frame['is_stale'] = feature_frame['days_since_update'] >= 180
feature_frame['ctr_position_gap'] = (
    feature_frame['position_band'].notna()
    & (feature_frame['ctr_mar'] < 0.5 * feature_frame['peer_median_ctr'])
)

feature_frame['score'] = (
    feature_frame['is_eligible'].astype(int)
    * feature_frame['is_stale'].astype(int)
    * feature_frame['ctr_position_gap'].astype(int)
    * feature_frame['gsc_impressions_mar']
)

feature_frame['reason_code'] = np.where(
    feature_frame['score'] > 0, 'stale_visible_ctr_gap', None
)
feature_frame['action_label'] = np.where(
    feature_frame['score'] > 0, 'review_for_refresh', 'no_action_flagged'
)

ranked_queue = feature_frame.sort_values('score', ascending=False).reset_index(drop=True)

n_flagged = (ranked_queue['score'] > 0).sum()
print(f'Rows flagged for review_for_refresh: {n_flagged} of {len(ranked_queue)} '
      f'({n_flagged / len(ranked_queue):.1%})')

output_cols = [
    'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action_label',
    'gsc_impressions_mar', 'gsc_clicks_mar', 'ctr_mar', 'avg_position_mar', 'peer_median_ctr',
    'days_since_update', 'content_age_days', 'is_eligible', 'is_declining_proxy',
]

import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Wrote work/outputs/baseline_action_score.csv')
ranked_queue[output_cols].head(10)

Rows flagged for review_for_refresh: 1 of 134086 (0.0%)
Wrote work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,score,reason_code,action_label,gsc_impressions_mar,gsc_clicks_mar,ctr_mar,avg_position_mar,peer_median_ctr,days_since_update,content_age_days,is_eligible,is_declining_proxy
0,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,stale_visible_ctr_gap,review_for_refresh,3670.0,1.0,0.000272,6.555793,0.001439,232,232,True,0
1,client_157ffe4d4a595515,content_88c03e8eb0d7089b,0.0,NaN,no_action_flagged,726.0,2.0,0.002755,8.169038,0.001439,0,71,False,0
2,client_157ffe4d4a595515,content_88fa1a42b0ed7612,0.0,NaN,no_action_flagged,168.0,1.0,0.005952,9.234822,0.001439,0,74,False,0
3,client_157ffe4d4a595515,content_88fe52bf9d05da18,0.0,NaN,no_action_flagged,85.0,0.0,0.000000,8.027095,0.001439,0,74,False,0
4,client_157ffe4d4a595515,content_89008f8712458245,0.0,NaN,no_action_flagged,48.0,0.0,0.000000,14.931818,0.001202,0,112,False,0
5,client_65de48885f4ef01b,content_6eff10aeca2e754c,0.0,NaN,no_action_flagged,11.0,1.0,0.090909,14.457143,0.001202,34,274,False,1
6,client_157ffe4d4a595515,content_8a36834ce72c12ff,0.0,NaN,no_action_flagged,30.0,0.0,0.000000,16.947917,0.001202,0,71,False,0
7,client_157ffe4d4a595515,content_88b1daa0918ed139,0.0,NaN,no_action_flagged,276.0,2.0,0.007246,5.428830,0.001439,0,46,False,0
8,client_65de48885f4ef01b,content_6f13c8cfc469f132,0.0,NaN,no_action_flagged,103.0,0.0,0.000000,8.122186,0.001439,34,265,False,0
9,client_65de48885f4ef01b,content_6f5dc3e8a618b283,0.0,NaN,no_action_flagged,89.0,0.0,0.000000,10.841270,0.001202,0,375,False,0


### 2.1 Evaluate at K — precision@K against the proxy label

`is_declining_proxy` is not a perfect ground truth for "deserves a refresh" (see the scope note in ML-04), but it's the best evidence I have for whether the rule's top picks are meaningfully different from a random page. I report precision@K next to the base rate, per the baseline skill's rule: a precision number means nothing without it.

In [26]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = ranked_queue['is_declining_proxy'].mean()
print(f'Base rate (share of ALL pages that are is_declining_proxy=1): {base_rate:.3f}')

for k in (10, 20, 50, 100):
    p_at_k = precision_at_k(
        ranked_queue['score'].values, ranked_queue['is_declining_proxy'].values, k
    )
    print(f'precision@{k}: {p_at_k:.3f}  (vs base rate {base_rate:.3f}, '
          f'lift = {p_at_k - base_rate:+.3f})')


Base rate (share of ALL pages that are is_declining_proxy=1): 0.201
precision@10: 0.100  (vs base rate 0.201, lift = -0.101)
precision@20: 0.300  (vs base rate 0.201, lift = +0.099)
precision@50: 0.200  (vs base rate 0.201, lift = -0.001)
precision@100: 0.130  (vs base rate 0.201, lift = -0.071)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Generated straight from the actual top 10 rows in `ranked_queue` — the specific numbers below will match whatever this run of the notebook produced, not a fixed example. The "what would make this wrong" line mirrors slide 11's own worked example (a fading page-C-style article that's actually just seasonal, not stale) — the top-ten review exists precisely to catch what the rule can't see.

In [27]:
top20 = ranked_queue.head(20).copy()

for i, row in top20.iterrows():
    confidence = (
        'high' if row['gsc_impressions_mar'] >= 2000
        else 'medium' if row['gsc_impressions_mar'] >= 500
        else 'low'
    )
    print(f"{i+1}. content_id={row['content_hash_id']}  |  action: {row['action_label']}")
    print(f"   why: reason_code={row['reason_code']} -- "
          f"{int(row['days_since_update'])}d since update, "
          f"{int(row['gsc_impressions_mar'])} impressions in March, "
          f"avg position {row['avg_position_mar']:.1f}, CTR {row['ctr_mar']*100:.2f}% "
          f"vs. its position band's peer median {row['peer_median_ctr']*100:.2f}% "
          f"-> score {int(row['score']):,}")
    print(f"   confidence: {confidence} (based on visibility volume this month)")
    print(f"   what would make this wrong: the page could be seasonally quiet right now rather than "
          f"stale (evergreen content with a natural low season -- the rule can't see a calendar, slide "
          f"11's exact warning), the CTR gap could come from a rich SERP feature stealing clicks rather "
          f"than a bad title/snippet, or a client-wide SERP/algorithm shift could be hitting many pages "
          f"at once and this one just got caught in it.")
    print()


1. content_id=content_bea86ce3455100b0  |  action: review_for_refresh
   why: reason_code=stale_visible_ctr_gap -- 232d since update, 3670 impressions in March, avg position 6.6, CTR 0.03% vs. its position band's peer median 0.14% -> score 3,670
   confidence: high (based on visibility volume this month)
   what would make this wrong: the page could be seasonally quiet right now rather than stale (evergreen content with a natural low season -- the rule can't see a calendar, slide 11's exact warning), the CTR gap could come from a rich SERP feature stealing clicks rather than a bad title/snippet, or a client-wide SERP/algorithm shift could be hitting many pages at once and this one just got caught in it.

2. content_id=content_88c03e8eb0d7089b  |  action: no_action_flagged
   why: reason_code=nan -- 0d since update, 726 impressions in March, avg position 8.2, CTR 0.28% vs. its position band's peer median 0.14% -> score 0
   confidence: medium (based on visibility volume this month)
   w

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [28]:
# --- Weak-pick scan: within the flagged set, the LOWEST-scoring flagged rows are where a
# borderline gate (barely over 500 impressions, barely under the CTR line) can let a weak pick through.
flagged = ranked_queue[ranked_queue['score'] > 0]
weakest_flagged = flagged.sort_values('score', ascending=True).head(5)
print('5 weakest rows that still passed all gates -- worth a skeptical look:')
weakest_flagged[['content_hash_id', 'score', 'gsc_impressions_mar', 'ctr_mar', 'peer_median_ctr',
                  'avg_position_mar', 'days_since_update', 'is_declining_proxy']]


5 weakest rows that still passed all gates -- worth a skeptical look:


,content_hash_id,score,gsc_impressions_mar,ctr_mar,peer_median_ctr,avg_position_mar,days_since_update,is_declining_proxy
0,content_bea86ce3455100b0,3670.0,3670.0,0.000272,0.001439,6.555793,232,0


In [29]:
# --- Leakage check: assert every score input is March-window-only and not label-derived. ---
assert MONTH == '2026-03', 'Feature window drifted off the mid-panel month.'

score_inputs = {'is_eligible', 'is_stale', 'ctr_position_gap', 'gsc_impressions_mar',
                'avg_position_mar', 'ctr_mar', 'peer_median_ctr', 'days_since_update',
                'content_age_days', 'optimization_eligible_date'}
forbidden = {'is_declining_proxy', 'gsc_impressions_feb', 'pct_change_mar_vs_feb',
             'health_score', 'priority_score', 'action_type', 'refresh_tier'}

leak = score_inputs & forbidden
assert not leak, f'Score uses forbidden/label-derived columns: {leak}'
print('OK: score built only from', sorted(score_inputs))
print('OK: none of', sorted(forbidden), 'were used as score inputs -- is_declining_proxy is')
print('    used only for the signal-check verdicts and the precision@K sanity check above, never')
print('    fed into the score itself.')
print('OK: no report_date beyond', MONTH, 'was queried -- April 2026 onward was never touched.')
print('OK: peer_median_ctr is computed from the SAME March window, not a future period.')


OK: score built only from ['avg_position_mar', 'content_age_days', 'ctr_mar', 'ctr_position_gap', 'days_since_update', 'gsc_impressions_mar', 'is_eligible', 'is_stale', 'optimization_eligible_date', 'peer_median_ctr']
OK: none of ['action_type', 'gsc_impressions_feb', 'health_score', 'is_declining_proxy', 'pct_change_mar_vs_feb', 'priority_score', 'refresh_tier'] were used as score inputs -- is_declining_proxy is
    used only for the signal-check verdicts and the precision@K sanity check above, never
    fed into the score itself.
OK: no report_date beyond 2026-03 was queried -- April 2026 onward was never touched.
OK: peer_median_ctr is computed from the SAME March window, not a future period.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.